# Управление потоком исполнения: if-else, граф, ReAct

Ноутбук к занятию 4 курса **«Разработка LLM-приложений с LangGraph»**.

На занятии 3 мы дошли до агента: в демо 6 написали цикл `run_with_tools`, а в бонусе увидели тот же цикл как граф в LangGraph.
Сегодня разбираем, **кто управляет потоком исполнения** — код разработчика или модель — и пробуем три подхода на одной задаче:

> «Прочитай все договоры в папке, найди просроченные и по каждому подготовь уведомление».

| Раздел | Что делаем | Демо на слайдах |
|---|---|---|
| 1 | Подготовка: доступ к модели и данные | — |
| 2 | Маршрутизатор на if-else | Демо 1 |
| 3 | Цепочка вызовов с проверкой (prompt chaining) | Демо 2 |
| 4 | Повтор и исправление вручную | Демо 3 |
| 5 | Тот же процесс как граф в LangGraph | Демо 4 |
| 6 | Checkpointer и подтверждение человеком | Демо 5 |
| 7 | ReAct-агент для задачи про договоры | Демо 6 |
| 8 | ReAct как граф с контролем | Демо 7 |
| 9 | Итог: сравниваем подходы | — |

Разделы 2–4 и 7 написаны на **чистом `requests`**, как на занятии 3. В разделах 5–6 LangGraph управляет потоком,
а внутри узлов — те же наши функции. В разделе 8 подключаем готовую интеграцию модели для LangChain.

> На весь ноутбук уходит около 50 запросов к модели — с `gpt-4o-mini` это несколько рублей.

## 1. Подготовка

Ячейки этого раздела повторяют подготовку из ноутбука занятия 3: ноутбук должен работать сам по себе.

### Где взять ключ
1. Зарегистрируйтесь на **aitunnel.ru** и пополните баланс.
2. В панели откройте раздел **«Ключи»** и создайте ключ. Он выглядит как `sk-aitunnel-…`. Для учёбы удобно задать ключу **бюджет**.

### Куда положить ключ
В Colab: значок 🔑 (**Secrets**) на левой панели → **Add new secret**:
- Name: `AITUNNEL_API_KEY`
- Value: ваш ключ `sk-aitunnel-…`
- включите переключатель **Notebook access**.

In [ ]:
# LangGraph понадобится с раздела 5, интеграция модели для LangChain — в разделе 8.
%pip install -q "langgraph>=1.0,<2" langchain-openai

In [ ]:
import os, json, time, re, ast, operator, datetime as dt
from contextlib import contextmanager
from pathlib import Path
import requests

# --- Ключ: из Colab Secrets, иначе из переменной окружения, иначе спросим ---
try:
    from google.colab import userdata
    API_KEY = userdata.get("AITUNNEL_API_KEY")
except Exception:
    from getpass import getpass
    API_KEY = os.environ.get("AITUNNEL_API_KEY") or getpass("AITunnel API key: ")
API_KEY = (API_KEY or "").strip()

# --- Вся конфигурация провайдера — в одной ячейке ---
API_URL = "https://api.aitunnel.ru/v1"        # если адрес недоступен из вашей сети: "https://ru-api.aitunnel.ru/v1"
MODEL   = "gpt-4o-mini"                       # дешёвая модель с хорошим вызовом инструментов
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

print("Ключ загружен:", API_KEY.startswith("sk-aitunnel-"))

### Обёртка `chat()` и учёт токенов

`chat()` — единственное место, где мы ходим в API. `measure()` считает, сколько вызовов модели и токенов ушло на каждый подход:
в конце ноутбука сравним их между собой.

In [ ]:
USAGE_LOG = []   # usage каждого ответа модели

def chat(messages, model=None, **params):
    """POST /chat/completions. Возвращает разобранный JSON ответа целиком."""
    payload = {"model": model or MODEL, "messages": messages, **params}
    for attempt in range(3):
        resp = requests.post(f"{API_URL}/chat/completions",
                             headers={**HEADERS, "Content-Type": "application/json"},
                             json=payload, timeout=120)
        if resp.status_code == 429 or resp.status_code >= 500:   # лимит или сбой провайдера — повторяем
            time.sleep(2 ** attempt)
            continue
        if not resp.ok:                      # 400, 401, 402 — повтор не поможет
            print("Ошибка API:", resp.status_code, resp.text[:500])
        resp.raise_for_status()
        data = resp.json()
        USAGE_LOG.append({"model": data.get("model", payload["model"]), **(data.get("usage") or {})})
        return data
    resp.raise_for_status()


try:
    catalog = requests.get("https://api.aitunnel.ru/public/aitunnel/models/chat", timeout=30).json()
except Exception:
    catalog = {}

def price(model):
    """Цена модели: (₽ за 1 млн входных токенов, ₽ за 1 млн выходных)."""
    info = catalog.get(model, {})
    return float(info.get("prompt_cost", 0)), float(info.get("completion_cost", 0))

def usage_report(log=None):
    log = USAGE_LOG if log is None else log
    prompt = sum(u.get("prompt_tokens", 0) for u in log)
    completion = sum(u.get("completion_tokens", 0) for u in log)
    cost = sum((u.get("prompt_tokens", 0) * price(u.get("model", MODEL))[0] +
                u.get("completion_tokens", 0) * price(u.get("model", MODEL))[1]) / 1_000_000 for u in log)
    print(f"Запросов: {len(log)}")
    print(f"Токены: запрос {prompt}, ответ {completion}, всего {prompt + completion}")
    print(f"Стоимость: {cost:.4f} ₽")

In [ ]:
def reply_text(data):
    """Достаёт текст ответа из JSON."""
    return data["choices"][0]["message"]["content"]

def ask(system, user, **params):
    """Один вызов модели: системный промпт + сообщение пользователя → текст ответа."""
    return reply_text(chat([{"role": "system", "content": system},
                            {"role": "user", "content": user}], **params))

def parse_json(text):
    """Достаёт первый JSON-объект из текста (модель иногда оборачивает его в ```json ... ```)."""
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"JSON не найден в ответе: {text[:200]}")
    return json.loads(match.group(0))


STATS = []   # сводка по подходам: вызовы модели, токены, время

@contextmanager
def measure(label):
    """Считает вызовы chat() и токены внутри блока with."""
    start, t0 = len(USAGE_LOG), time.time()
    yield
    part = USAGE_LOG[start:]
    STATS.append({"подход": label, "вызовов LLM": len(part),
                  "токенов": sum(u.get("total_tokens", 0) for u in part),
                  "секунд": round(time.time() - t0, 1)})
    print(f"\n[{label}] вызовов LLM: {len(part)}, токенов: {STATS[-1]['токенов']}")

print(ask("Отвечай одним словом.", "Столица Франции?"))

### Данные для занятия

1. **Мини-база правил университета** из занятия 3 — для маршрутизатора в разделе 2.
2. **Папка `contracts/` с пятью договорами** — для всех остальных разделов. Даты считаются от сегодняшнего дня, поэтому
   договоры 101 и 103 всегда просрочены, 102 и 105 действуют, а в 104 срока нет совсем — на нём будем ломать извлечение.

In [ ]:
KB = [
    {"id": "retake", "text": "Пересдача экзамена проводится не позднее двух месяцев после окончания сессии. "
                             "Первая пересдача принимается преподавателем, вторая — комиссией. Запись через деканат."},
    {"id": "stipend", "text": "Академическая стипендия назначается по итогам сессии студентам, сдавшим экзамены "
                              "на «хорошо» и «отлично». Социальная стипендия назначается по заявлению в деканат."},
    {"id": "dorm", "text": "Места в общежитии предоставляются иногородним студентам. Заявление подаётся в студенческий совет до 15 августа."},
    {"id": "coursework", "text": "Курсовая работа защищается до начала зачётной недели. Тему согласуют с руководителем не позднее октября."},
    {"id": "contract", "text": "Договор на платное обучение подписывается в отделе по работе со студентами, кабинет 105. "
                               "Оплата вносится по семестрам до 1 сентября и до 1 февраля."},
    {"id": "library", "text": "Библиотека работает с 9:00 до 19:00 по будням. Учебники выдаются на семестр по студенческому билету."},
]

def tokenize(text):
    return {w for w in re.findall(r"[а-яёa-z0-9]+", text.lower()) if len(w) > 3}

def search_kb(query, k=2):
    """Поиск по совпадению слов — тот же наивный RAG, что на занятии 3."""
    q = tokenize(query)
    scored = sorted(((len(q & tokenize(d["text"])), d) for d in KB), key=lambda x: -x[0])
    return {"results": [d for score, d in scored[:k] if score > 0]}


TODAY = dt.date.today()

def day(offset):
    """Дата через offset дней от сегодня в формате ДД.ММ.ГГГГ."""
    return (TODAY + dt.timedelta(days=offset)).strftime("%d.%m.%Y")

CONTRACTS = {
    "d-101.txt": f"""ДОГОВОР ОКАЗАНИЯ УСЛУГ № 101
Заказчик: ООО «Ромашка». Исполнитель: ИП Иванов Иван Иванович.
Исполнитель обязуется оказать услуги по настройке учётной системы.
Стоимость услуг: 120 000 (сто двадцать тысяч) рублей.
Срок действия договора: до {day(-75)}.""",
    "d-102.txt": f"""ДОГОВОР ПОСТАВКИ № 102
Поставщик: АО «Север». Покупатель: ООО «Вектор».
Поставщик передаёт покупателю офисную мебель согласно спецификации.
Общая стоимость поставки: 450 000 рублей.
Договор действует до {day(200)}.""",
    "d-103.txt": f"""ДОГОВОР АРЕНДЫ № 103
Арендодатель: ИП Петрова Анна Сергеевна. Арендатор: ООО «Бета».
Предмет: нежилое помещение площадью 40 кв. м.
Арендная плата: 35 000 рублей в месяц.
Срок аренды: до {day(-12)} включительно.""",
    "d-104.txt": """ДОГОВОР ПОДРЯДА № 104
Заказчик: ООО «Гамма». Подрядчик: ООО «СтройМастер».
Подрядчик выполняет ремонт помещения по заданию заказчика.
Стоимость работ определяется сметой (Приложение 1).
Срок действия договора устанавливается дополнительным соглашением сторон.""",
    "d-105.txt": f"""ДОГОВОР ОБ ОКАЗАНИИ ПЛАТНЫХ ОБРАЗОВАТЕЛЬНЫХ УСЛУГ № 105
Исполнитель: Университет. Заказчик: Сидоров Пётр Алексеевич.
Стоимость обучения за год: сто восемьдесят тысяч рублей.
Договор действует до {day(40)}.""",
}

Path("contracts").mkdir(exist_ok=True)
for name, text in CONTRACTS.items():
    Path("contracts", name).write_text(text, encoding="utf-8")

print("Сегодня:", TODAY.strftime("%d.%m.%Y"))
for name, text in CONTRACTS.items():
    print(f"  {name}: {text.splitlines()[-1]}")

## 2. Демо 1 — маршрутизатор на if-else

Слайды 6–7. Модель только **классифицирует** вопрос и возвращает метку. Какие шаги выполнить для каждой метки
и в каком порядке, решает наш код — обычный `if / elif`.

In [ ]:
CLASSIFY_SYSTEM = """Ты классификатор вопросов студента. Верни ОДНО слово — метку:
rules — вопрос о правилах университета (пересдачи, стипендия, общежитие, курсовые, договор на обучение);
calc — нужно что-то посчитать;
chitchat — приветствие или разговор не по делу.
Только метку, без пояснений."""

RAG_SYSTEM = "Ты справочный бот университета. Отвечай кратко и только по контексту. Если ответа в нём нет — так и скажи."

EXPR_SYSTEM = "Переведи задачу в одно арифметическое выражение на Python. Верни ТОЛЬКО выражение, без слов и без знака =."

_ALLOWED_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
                ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def _safe_eval(node):
    """Вычисляет только арифметику: eval() на выводе модели — дыра в безопасности."""
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Разрешена только арифметика")

def calculator(expression):
    return {"expression": expression, "result": round(_safe_eval(ast.parse(expression, mode="eval")), 6)}


def classify(question):
    label = ask(CLASSIFY_SYSTEM, question, temperature=0.1).strip().lower()
    label = re.sub(r"[^a-z]", "", label)
    return label if label in {"rules", "calc", "chitchat"} else "unknown"

def rag_answer(question, hits):
    context = "\n".join(f"[{h['id']}] {h['text']}" for h in hits) or "(ничего не найдено)"
    return ask(RAG_SYSTEM, f"Контекст:\n{context}\n\nВопрос: {question}", temperature=0.1)

def extract_expression(question):
    return ask(EXPR_SYSTEM, question, temperature=0.1).strip().strip("`").replace(",", ".")


def answer_question(question):
    intent = classify(question)                        # LLM возвращает метку
    print(f"  метка: {intent}")
    if intent == "rules":
        hits = search_kb(question)["results"]
        return rag_answer(question, hits)              # ещё один вызов LLM
    elif intent == "calc":
        expr = extract_expression(question)            # LLM → строка
        print(f"  выражение: {expr}")
        try:
            return calculator(expr)["result"]
        except Exception as e:
            return f"Не смог посчитать: {e}"
    elif intent == "chitchat":
        return reply_text(chat([{"role": "user", "content": question}]))
    else:
        return "Не понял вопрос"


with measure("if-else: маршрутизатор"):
    for q in ["Кто принимает вторую пересдачу?",
              "Сколько будет 17,5% от 2340 рублей?",
              "Привет! Как настроение?"]:
        print(f"Вопрос: {q}")
        print(f"Ответ: {answer_question(q)}\n")

Поток полностью предсказуем: на каждый вопрос ровно два вызова модели, путь виден в коде.

Теперь дадим маршрутизатору задачу занятия. Какую ветку он выберет?

In [ ]:
answer_question("Прочитай все договоры в папке contracts, найди просроченные и по каждому подготовь уведомление.")

Ни одна ветка не подходит: в задаче заранее неизвестно, **сколько** файлов читать и **сколько** уведомлений писать.
Чтобы её решить, нужен цикл «делай, пока не готово» — а его в маршрутизаторе нет.

**Попробуйте сами:** добавьте метку `deadline` («сколько дней до …») и ветку для неё. Сколько строк кода пришлось поменять?

## 3. Демо 2 — цепочка вызовов с проверкой

Слайд 9, паттерн **prompt chaining**. Разберём один договор так же, как в практике занятия 2, но разобьём работу на шаги:

1. **LLM** извлекает поля в JSON;
2. **код** проверяет их (gate) — если что-то не так, цепочка останавливается;
3. **LLM** оформляет уведомление, если договор просрочен.

Путь по-прежнему задан кодом: шаги идут строго друг за другом.

In [ ]:
EXTRACT_SYSTEM = """Ты извлекаешь данные из договора. Ответь ТОЛЬКО JSON без пояснений:
{"number": номер договора строкой,
 "kind": вид договора (услуги, поставка, аренда и т.п.),
 "parties": [стороны договора],
 "amount": сумма в рублях числом или null,
 "end_date": дата окончания действия в формате YYYY-MM-DD или null}
Если чего-то нет в тексте — ставь null, ничего не придумывай."""

NOTICE_SYSTEM = ("Ты юрист. Напиши короткое уведомление (2–3 предложения) второй стороне о том, "
                 "что срок действия договора истёк. Без подписи и без шаблонов в квадратных скобках.")

def read_contract(path):
    return Path(path).read_text(encoding="utf-8")

def extract_fields(text):
    return parse_json(ask(EXTRACT_SYSTEM, text, temperature=0.1))

def validate(fields):
    """Gate: обычный код, никакой модели."""
    errors = []
    if not fields.get("number"):
        errors.append("нет номера договора")
    if len(fields.get("parties") or []) < 2:
        errors.append("должно быть две стороны")
    amount = fields.get("amount")
    if not isinstance(amount, (int, float)) or amount <= 0:
        errors.append("amount должна быть положительным числом")
    try:
        dt.date.fromisoformat(str(fields.get("end_date")))
    except ValueError:
        errors.append("end_date должна быть датой YYYY-MM-DD")
    return errors

def make_notice(fields):
    return ask(NOTICE_SYSTEM, json.dumps(fields, ensure_ascii=False), temperature=0.3)


def chain(path):
    fields = extract_fields(read_contract(path))       # LLM 1: извлечь
    print("  поля:", fields)
    errors = validate(fields)                          # gate: проверить кодом
    if errors:
        return f"⛔ цепочка остановлена на проверке: {errors}"
    if dt.date.fromisoformat(fields["end_date"]) >= TODAY:
        return "договор действует — уведомление не нужно"
    return make_notice(fields)                         # LLM 2: оформить


with measure("цепочка с проверкой"):
    for path in ["contracts/d-101.txt", "contracts/d-104.txt"]:
        print(f"📄 {path}")
        print(chain(path), "\n")

Договор 104 не прошёл проверку: в нём нет ни суммы, ни срока. Цепочка честно остановилась — но сама исправить ничего не может.

**Попробуйте сами:** запустите `chain("contracts/d-105.txt")`. Сумма там написана словами — справилась ли модель с переводом в число?

## 4. Демо 3 — повтор и исправление вручную

Слайды 11 и 15. Добавим шаг «исправить»: если проверка не прошла, просим модель поправить JSON с учётом ошибок
и проверяем снова — не больше трёх раз, потом отдаём человеку.

In [ ]:
FIX_SYSTEM = """Ты исправляешь JSON с данными договора. Тебе дадут текст договора, текущий JSON и список ошибок.
Верни ТОЛЬКО исправленный JSON той же структуры. Если данных в тексте нет — оставь null, не выдумывай."""

def fix_fields(text, fields, errors):
    user = f"Текст договора:\n{text}\n\nТекущий JSON:\n{json.dumps(fields, ensure_ascii=False)}\n\nОшибки: {errors}"
    try:
        return parse_json(ask(FIX_SYSTEM, user, temperature=0.1))
    except ValueError:
        return fields                                  # модель ответила не JSON — оставляем как было


def chain_with_retry(path, max_attempts=3):
    text = read_contract(path)
    fields = extract_fields(text)
    attempts = 0
    while True:                                        # ← цикл «пока не готово»
        errors = validate(fields)
        if not errors:
            return f"✅ поля прошли проверку: {fields}"
        if attempts >= max_attempts:
            return f"🙋 нужен человек: {errors}"
        attempts += 1
        print(f"  попытка {attempts}: {errors}")
        fields = fix_fields(text, fields, errors)


with measure("цепочка с повтором"):
    print(chain_with_retry("contracts/d-104.txt"))

Это уже **граф**, только нарисованный неявно: узлы «извлечь», «проверить», «исправить», ребро назад и условие выхода по счётчику.
Пока код маленький, это не страшно. Но как только понадобится остановиться и подождать человека, продолжить после сбоя
или посмотреть, на каком шаге мы сейчас, всё это придётся писать самим. Дальше то же самое опишем явно.

## 5. Демо 4 — тот же процесс как граф в LangGraph

Слайды 14, 16 и 17. Пять понятий:

- **состояние** `DocState` — общая структура для всех узлов;
- **узел** — функция «состояние → обновление состояния»;
- **ребро** — безусловный переход;
- **условное ребро** — функция `route()` выбирает следующий узел;
- **цикл** — ребро `fix → validate`.

Внутри узлов — **те же функции** из разделов 3–4 на `requests`. LangGraph управляет только потоком: ему всё равно,
как именно узел обращается к модели.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

MAX_ATTEMPTS = 3
Path("extracted").mkdir(exist_ok=True)

class DocState(TypedDict, total=False):
    path: str
    text: str
    fields: dict
    errors: list[str]
    attempts: int
    status: str

def extract(state: DocState) -> dict:                  # LLM
    text = read_contract(state["path"])
    return {"text": text, "fields": extract_fields(text), "attempts": 0}

def validate_node(state: DocState) -> dict:            # код
    return {"errors": validate(state["fields"])}

def fix(state: DocState) -> dict:                      # LLM
    return {"fields": fix_fields(state["text"], state["fields"], state["errors"]),
            "attempts": state["attempts"] + 1}

def save(state: DocState) -> dict:                     # код
    out = Path("extracted", Path(state["path"]).stem + ".json")
    out.write_text(json.dumps(state["fields"], ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status": f"сохранено в {out}"}

def to_human(state: DocState) -> dict:                 # код
    return {"status": f"нужен человек: {state['errors']}"}

def route(state: DocState) -> str:                     # условное ребро: решает код
    if not state["errors"]:
        return "ok"
    if state["attempts"] < MAX_ATTEMPTS:
        return "retry"
    return "human"


g = StateGraph(DocState)
g.add_node("extract", extract)
g.add_node("validate", validate_node)
g.add_node("fix", fix)
g.add_node("save", save)
g.add_node("human", to_human)

g.add_edge(START, "extract")
g.add_edge("extract", "validate")
g.add_conditional_edges("validate", route, {"ok": "save", "retry": "fix", "human": "human"})
g.add_edge("fix", "validate")
g.add_edge("save", END)
g.add_edge("human", END)

doc_app = g.compile()

In [ ]:
# Граф можно нарисовать: сверяем с картинкой на слайде 16
from IPython.display import Image, display

def draw(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())   # если рендер недоступен — текст в Mermaid

draw(doc_app)

Запускаем через `stream`: граф отдаёт обновление после каждого узла, и мы видим путь исполнения.

In [ ]:
def short(update):
    """Краткое описание обновления состояния: без полного текста договора."""
    shown = {k: v for k, v in (update or {}).items() if k != "text"}
    return json.dumps(shown, ensure_ascii=False, default=str)[:150]

def run_graph(app, inputs, config=None):
    for step in app.stream(inputs, config, stream_mode="updates"):
        for node, update in step.items():
            if node == "__interrupt__":
                print("  ⏸ граф остановился и ждёт человека")
            else:
                print(f"  → {node}: {short(update)}")


with measure("граф: обработка договора"):
    for path in ["contracts/d-101.txt", "contracts/d-104.txt"]:
        print(f"📄 {path}")
        run_graph(doc_app, {"path": path})
        print()

Поведение то же, что в разделе 4, но поток теперь описан явно: его можно нарисовать, у каждого узла понятный вход и выход,
а `route()` легко протестировать отдельно.

**Попробуйте сами:** поменяйте `MAX_ATTEMPTS` на 1 и посмотрите, как изменился путь для договора 104.

## 6. Демо 5 — checkpointer и подтверждение человеком

Слайд 19: что граф даёт сверх обычного цикла.

- **Checkpointer** сохраняет состояние после каждого шага. Граф можно остановить и продолжить, посмотреть историю шагов.
- **`interrupt()`** ставит граф на паузу внутри узла и ждёт решения человека. Продолжаем командой `Command(resume=...)`.

Добавим перед сохранением узел `approve`: человек видит извлечённые поля и решает, сохранять ли их.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

def approve(state: DocState) -> dict:
    """Пауза: граф останавливается и ждёт решения человека."""
    decision = interrupt({"вопрос": "Сохранить извлечённые поля?", "поля": state["fields"]})
    return {"status": "одобрено" if decision == "да" else "отклонено человеком"}

def after_approve(state: DocState) -> str:
    return "save" if state["status"] == "одобрено" else "end"


g2 = StateGraph(DocState)
for name, fn in [("extract", extract), ("validate", validate_node), ("fix", fix),
                 ("approve", approve), ("save", save), ("human", to_human)]:
    g2.add_node(name, fn)

g2.add_edge(START, "extract")
g2.add_edge("extract", "validate")
g2.add_conditional_edges("validate", route, {"ok": "approve", "retry": "fix", "human": "human"})
g2.add_edge("fix", "validate")
g2.add_conditional_edges("approve", after_approve, {"save": "save", "end": END})
g2.add_edge("save", END)
g2.add_edge("human", END)

doc_app2 = g2.compile(checkpointer=MemorySaver())   # без checkpointer пауза невозможна
draw(doc_app2)

In [ ]:
# thread_id — «номер разговора»: по нему checkpointer находит сохранённое состояние
config = {"configurable": {"thread_id": "contract-102"}}

print("📄 contracts/d-102.txt")
run_graph(doc_app2, {"path": "contracts/d-102.txt"}, config)

snapshot = doc_app2.get_state(config)
print("\nСледующий узел:", snapshot.next)
print("Граф спрашивает:", snapshot.tasks[0].interrupts[0].value)

Граф стоит на паузе, а его состояние лежит в checkpointer. Можно закрыть ноутбук на час — с `thread_id` мы продолжим с того же места
(в учебном `MemorySaver` состояние живёт в памяти; в продакшене берут checkpointer на базе данных).

Отвечаем «да» или «нет». Чтобы ячейка не ждала ввода при запуске всего ноутбука, задайте `AUTO_DECISION = "да"`.

In [ ]:
AUTO_DECISION = None   # None — спросить через input(); "да" / "нет" — ответить автоматически

decision = AUTO_DECISION or input("Сохранить поля? (да/нет): ").strip().lower()
run_graph(doc_app2, Command(resume=decision), config)
print("Статус:", doc_app2.get_state(config).values.get("status"))

In [ ]:
# История шагов из checkpointer: от последнего к первому
for snap in doc_app2.get_state_history(config):
    print(f"шаг {snap.metadata.get('step')}: следующий узел {snap.next}, статус {snap.values.get('status')}")

Связь с занятием 3: checkpointer — это «память» без ручного хранения `messages`, а подтверждение человеком — последний рубеж
защиты, когда инъекция может превратиться в действие.

**Попробуйте сами:** запустите тот же граф для `d-101.txt` с другим `thread_id` и ответьте «нет». Какой путь прошёл граф?

## 7. Демо 6 — ReAct-агент для задачи про договоры

Слайды 23–26. Граф хорош, когда путь известен. В нашей задаче он неизвестен: сколько договоров, какие просрочены, сколько уведомлений —
выясняется по ходу. Отдадим выбор следующего шага модели.

Инструменты агента — обычные функции:
`list_files`, `read_document`, `get_current_date`, `days_between` и `save_note`.

In [ ]:
def list_files(folder="contracts"):
    p = Path(folder)
    if not p.is_dir():
        raise FileNotFoundError(f"папка {folder} не найдена")
    return {"files": sorted(str(f) for f in p.iterdir() if f.is_file())}

def read_document(path):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"файл {path} не найден")
    return {"path": str(p), "text": p.read_text(encoding="utf-8")}

def get_current_date():
    return {"date": dt.date.today().isoformat()}

def days_between(date_from, date_to):
    return {"days": (dt.date.fromisoformat(date_to) - dt.date.fromisoformat(date_from)).days}

def save_note(name, text):
    safe = re.sub(r"[^\w\-]", "_", name)[:60]
    out = Path("notes", f"{safe}.txt")
    out.write_text(text, encoding="utf-8")
    return {"saved": str(out)}

AGENT_TOOLS = {"list_files": list_files, "read_document": read_document, "get_current_date": get_current_date,
               "days_between": days_between, "save_note": save_note}

FUNCTIONS = [
    {"name": "list_files", "description": "Возвращает список файлов в папке.",
     "parameters": {"type": "object",
                    "properties": {"folder": {"type": "string", "description": "Папка, например contracts"}},
                    "required": ["folder"]}},
    {"name": "read_document", "description": "Читает текст договора по пути к файлу.",
     "parameters": {"type": "object",
                    "properties": {"path": {"type": "string", "description": "Путь, например contracts/d-101.txt"}},
                    "required": ["path"]}},
    {"name": "get_current_date", "description": "Возвращает сегодняшнюю дату в формате YYYY-MM-DD.",
     "parameters": {"type": "object", "properties": {}}},
    {"name": "days_between", "description": "Считает количество дней между двумя датами. Отрицательное число — вторая дата раньше первой.",
     "parameters": {"type": "object",
                    "properties": {"date_from": {"type": "string", "description": "Дата начала, YYYY-MM-DD"},
                                   "date_to": {"type": "string", "description": "Дата конца, YYYY-MM-DD"}},
                    "required": ["date_from", "date_to"]}},
    {"name": "save_note", "description": "Сохраняет текст уведомления в файл.",
     "parameters": {"type": "object",
                    "properties": {"name": {"type": "string", "description": "Имя уведомления, например notice-101"},
                                   "text": {"type": "string", "description": "Текст уведомления"}},
                    "required": ["name", "text"]}},
]

AGENT_SYSTEM = ("Ты помощник юриста. Работай только через функции: смотри список файлов, читай договоры, "
                "узнавай сегодняшнюю дату. Даты в договорах записаны как ДД.ММ.ГГГГ. "
                "Уведомления сохраняй функцией save_note. Когда всё сделано — кратко перечисли, что сделано.")

### Цикл агента

Это `run_with_tools` из демо 6 занятия 3 с тремя доработками со слайда 26:
печатаем траекторию в терминах ReAct, сохраняем все вызовы в `TRACE` и останавливаемся, если модель повторяет тот же вызов.

Напоминание о формате OpenAI-совместимого API: инструменты передаём в поле `tools`, вызовы приходят списком `message.tool_calls`,
аргументы — строкой с JSON, результат возвращаем сообщением с ролью `tool` и `tool_call_id`.

In [ ]:
TRACE = []   # все вызовы инструментов последнего запуска
TOOL_SPECS = [{"type": "function", "function": f} for f in FUNCTIONS]

def run_agent(task, max_steps=15, verbose=True):
    TRACE.clear()
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": task}]

    for step in range(1, max_steps + 1):
        data = chat(messages, tools=TOOL_SPECS, tool_choice="auto")
        msg = data["choices"][0]["message"]
        messages.append(msg)                                  # ответ модели — в историю целиком

        calls = msg.get("tool_calls") or []
        if not calls:
            return msg["content"]                             # Final Answer: модель решила, что готово

        if verbose and msg.get("content"):
            print(f"Thought: {msg['content']}")
        for call in calls:                                    # за один шаг вызовов может быть несколько
            name = call["function"]["name"]
            try:
                args = json.loads(call["function"].get("arguments") or "{}")
            except json.JSONDecodeError as e:
                args, result = {}, {"error": f"аргументы не JSON: {e}"}
            else:
                if TRACE and TRACE[-1] == (name, args):       # зацикливание: тот же вызов подряд
                    return f"⚠ Остановились: модель повторяет вызов {name}"
                TRACE.append((name, args))
                try:
                    result = AGENT_TOOLS[name](**args)        # Action выполняет наш код
                except Exception as e:
                    result = {"error": str(e)}                # ошибка — тоже наблюдение
            if verbose:
                print(f"Action: {name}({json.dumps(args, ensure_ascii=False)})")
                print(f"Observation: {json.dumps(result, ensure_ascii=False)[:150]}")
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": json.dumps(result, ensure_ascii=False)})

    return "⚠ Превышен лимит шагов"

In [ ]:
TASK = ("В папке contracts лежат договоры. Найди те, у которых срок действия уже истёк на сегодня, "
        "и по каждому сохрани короткое уведомление через save_note с именем notice-<номер договора>. "
        "В конце перечисли, что сделано.")

notes = Path("notes")
notes.mkdir(exist_ok=True)
for f in notes.glob("*.txt"):
    f.unlink()                                       # начинаем с пустой папки

start = len(USAGE_LOG)
with measure("ReAct: агент по договорам"):
    answer = run_agent(TASK)
print("\nFinal Answer:", answer)

In [ ]:
reads = [args for name, args in TRACE if name == "read_document"]
print(f"read_document вызван {len(reads)} раз — этого числа нет нигде в коде, его определил list_files")
print("Сохранённые уведомления:", sorted(p.name for p in Path("notes").glob("*.txt")))

# Рост контекста: каждый шаг отправляет всю историю заново
print("Токенов в запросе по шагам:", [u.get("prompt_tokens") for u in USAGE_LOG[start:]])

Обратите внимание на последнюю строку: каждый шаг агента дороже предыдущего, потому что история растёт. Это та же «цена памяти»,
что на занятии 3.

### Инженерные вопросы цикла

Три проверки со слайда 26. **Ошибка инструмента** — модель получает её как наблюдение и может отреагировать:

In [ ]:
print(run_agent("Прочитай договор contracts/d-999.txt и назови его сумму."))

**Лимит шагов** — защита от бесконечного цикла и перерасхода. С `max_steps=2` агент не успеет закончить работу:

In [ ]:
print(run_agent(TASK, max_steps=2))

**Зацикливание** — если модель дважды подряд делает тот же вызов, `run_agent` останавливается (проверка `TRACE[-1]`).
Вызвать это специально трудно: обычно случается на слабых моделях и длинных задачах.

**Попробуйте сами:** уберите из `AGENT_SYSTEM` фразу про формат дат и запустите задачу ещё раз. Ошибается ли модель с просрочкой?

## 8. Демо 7 — ReAct как граф с контролем

Слайды 29 и 33. Цикл из раздела 7 — это граф из двух узлов: `agent` вызывает модель, `tools` выполняет инструменты,
условное ребро `tools_condition` решает, идти ли дальше. Мы видели его в бонусе занятия 3.

Раз это граф, в него можно встроить контроль, которого нет в голом цикле. Сделаем так, чтобы **`save_note` не выполнялся без
подтверждения человека**: внутри инструмента вызываем `interrupt()`.

Здесь вместо `requests` используем готовую интеграцию модели для LangChain — так удобнее связывать модель с инструментами.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url=API_URL, api_key=API_KEY, model=MODEL, temperature=0)
BIND_KWARGS = {"parallel_tool_calls": False}   # по одному вызову за шаг: так проще подтверждать сохранение

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# Оборачиваем те же функции: docstring и аннотации типов становятся описанием для модели
@tool("list_files")
def lg_list_files(folder: str = "contracts") -> dict:
    """Возвращает список файлов в папке."""
    return list_files(folder)

@tool("read_document")
def lg_read_document(path: str) -> dict:
    """Читает текст договора по пути к файлу, например contracts/d-101.txt."""
    return read_document(path)

@tool("get_current_date")
def lg_current_date() -> dict:
    """Возвращает сегодняшнюю дату в формате YYYY-MM-DD."""
    return get_current_date()

@tool("days_between")
def lg_days_between(date_from: str, date_to: str) -> dict:
    """Считает количество дней между двумя датами в формате YYYY-MM-DD."""
    return days_between(date_from, date_to)

@tool("save_note")
def lg_save_note(name: str, text: str) -> dict:
    """Сохраняет текст уведомления в файл с именем name."""
    decision = interrupt({"действие": "save_note", "name": name, "text": text})   # ← ждём человека
    if decision != "да":
        return {"error": "человек отклонил сохранение"}
    return save_note(name, text)

lg_tools = [lg_list_files, lg_read_document, lg_current_date, lg_days_between, lg_save_note]
llm_with_tools = llm.bind_tools(lg_tools, **BIND_KWARGS)

def agent(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([SystemMessage(AGENT_SYSTEM)] + state["messages"])]}

g3 = StateGraph(MessagesState)
g3.add_node("agent", agent)
g3.add_node("tools", ToolNode(lg_tools))
g3.add_edge(START, "agent")
g3.add_conditional_edges("agent", tools_condition)   # есть вызовы → tools, иначе → END
g3.add_edge("tools", "agent")

react_app = g3.compile(checkpointer=MemorySaver())
draw(react_app)

`recursion_limit` в конфиге — тот же лимит шагов, что `max_steps` в разделе 7: при превышении LangGraph выбросит `GraphRecursionError`.

In [ ]:
def run_react(task, thread_id, limit=40):
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": limit}
    inputs = {"messages": [("user", task)]}
    t0 = time.time()
    while True:
        for step in react_app.stream(inputs, config, stream_mode="updates"):
            for node, update in step.items():
                if node == "__interrupt__":
                    continue
                for m in update["messages"]:
                    if getattr(m, "tool_calls", None):
                        for c in m.tool_calls:
                            print(f"Action: {c['name']}({json.dumps(c['args'], ensure_ascii=False)})")
                    elif m.type == "tool":
                        print(f"Observation: {str(m.content)[:150]}")
                    else:
                        print(f"\nFinal Answer: {m.content}")

        snapshot = react_app.get_state(config)
        if not snapshot.next:                        # граф дошёл до END
            break
        request = snapshot.tasks[0].interrupts[0].value
        print(f"\n⏸ Агент хочет сохранить {request['name']}:\n{request['text']}\n")
        decision = AUTO_DECISION or input("Разрешить? (да/нет): ").strip().lower()
        inputs = Command(resume=decision)            # продолжаем с того же места

    ai = [m for m in snapshot.values["messages"] if m.type == "ai"]
    STATS.append({"подход": "ReAct-граф с подтверждением", "вызовов LLM": len(ai),
                  "токенов": sum((m.usage_metadata or {}).get("total_tokens", 0) for m in ai),
                  "секунд": round(time.time() - t0, 1)})


for f in Path("notes").glob("*.txt"):
    f.unlink()
run_react(TASK, thread_id="react-1")
print("\nСохранённые уведомления:", sorted(p.name for p in Path("notes").glob("*.txt")))

Модель по-прежнему сама выбирает путь, но опасное действие проходит через человека. Так выглядит типичная продакшен-архитектура
со слайда 33: детерминированный каркас и автономия внутри одного узла.

**Попробуйте сами:** на первом запросе ответьте «нет». Что агент сделает дальше — попробует снова, пропустит договор или сообщит об отказе?

## 9. Итог: сравниваем подходы

Сколько вызовов модели и токенов ушло на каждый подход в этом запуске:

In [ ]:
import pandas as pd
pd.DataFrame(STATS)

| | if-else | Граф | ReAct |
|---|---|---|---|
| Кто выбирает следующий шаг | код | код + LLM на условных рёбрах | LLM |
| Число шагов | фиксировано | ограничено топологией | динамическое |
| Предсказуемость и тестируемость | высокая | средняя или высокая | низкая |
| Стоимость | минимальная | предсказуемая | плавающая |
| Наш пример | маршрутизатор вопросов | обработка одного договора | все договоры в папке |

Главное правило: **минимальная автономия, достаточная для задачи**. Маршрутизатор не справился с задачей про папку договоров,
а агент для одного договора был бы лишним — граф там проще, дешевле и предсказуемее.

**Домашнее задание:** нарисуйте поток управления своей курсовой — узлы, переходы и места, где решает модель.
Какие действия в ней должны проходить через подтверждение человека?

In [ ]:
usage_report()   # сколько токенов ушло за весь ноутбук